# Cell2location deconvolution — results visualisation and comparison vs. Ma et al RCTD results
* This notebook helps to visualize each deconvolution referenece strategy results. Select one in the dropdown menus below.
* For a comparison of the three strategies against each other see `notebooks/compare_reference.ipynb`.
* Inputs are the outputs of `reference_building/benchmark/reference_data/ma/scripts/03_build_benchmark.py`, `06_submit_cell2location_RefStrategy*sh` and `07_agreement_matrix_reference_RefStrategy*.py`

## Imports

In [ ]:
import glob
import sys
import warnings
from pathlib import Path

from IPython.display import Image, HTML, display
import ipywidgets as widgets

import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype"
sc.set_figure_params(dpi=100, frameon=False)
warnings.filterwarnings("ignore", message=".*deprecated.*")

## Inputs

In [ ]:
repo_name = "spatial-sc-rnaseq-integration-wf"  # needed for get_repo_root to work in both unpromoted and live repo notebooks
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from notebook_utils import get_repo_root # must come after the utils sys.path.insert above
repo_root = get_repo_root(repo_name)
BASE_INDIR = str(repo_root / "reference_building" / "benchmark" / "reference_data" / "ma" / "data" / "reformatted")

STRATEGIES = (1, 2, 3)
STRATEGY_DIR = {n: f"cell2location_Strategy{n}" for n in STRATEGIES}
H5AD_SUFFIX = ".deconvoluted_clustered.h5ad"
BASE_STRATEGY = 3  # all 3 strategies cover the same 10 Ma samples, so sample discovery only needs one

available_samples = sorted(
    Path(p).name.removesuffix(H5AD_SUFFIX)
    for p in glob.glob(str(Path(BASE_INDIR, STRATEGY_DIR[BASE_STRATEGY], f"*{H5AD_SUFFIX}")))
)
print(f"Found {len(available_samples)} sample(s) in {BASE_INDIR}: {available_samples}")

## Utils

In [ ]:
sys.path.insert(0, str(repo_root / "utils"))
from genomics_constants import KAMATH_DA_MARKER_SYMBOLS
from notebook_utils import (
    build_all_vs_all_figure,
    coarse_category as uber_type_of,
    dropdown,
    golden_angle_palette,
    inject_big_dropdown_css,
    load_all_vs_all,
    load_kamath_collapse_and_palette,
    selected_cell_types,
    spatial_scatter_grid,
    strip_q05_abundance_prefix,
)

inject_big_dropdown_css()

DA_UBER_CATEGORIES = {"CALB1", "SOX6"}

STRATEGY_OPTIONS = {f"Strategy {n}": n for n in STRATEGIES}
strategy_dropdown = dropdown(big=True, options=list(STRATEGY_OPTIONS), description="Strategy:")

sample_dropdown = dropdown(big=True, options=available_samples, description="Sample:")

# Used by selected_cell_types(), which sections 4 and 5 call to restrict to CALB1_*/SOX6_* cell types.
cell_type_view_dropdown = dropdown(big=True, options=["All cell types", "DAs only"], description="Show:")


# Shared collapse dicts/helpers from the Ma comparison script, loaded once.
_compare_mod, uber_categories, uber_palette = load_kamath_collapse_and_palette(repo_root)


def _ensure_uber_colors(cols):
    """
    Extend `uber_palette` in place with any uber category in `cols` not
    already covered.

    Parameters
    ----------
    cols : iterable of str
        Fine-grained cell type names (e.g. this load's `cell_types_`).

    Returns
    -------
    None
    """
    missing = sorted({uber_type_of(c) for c in cols} - set(uber_palette))
    if missing:
        uber_palette.update(golden_angle_palette(list(uber_palette) + missing))


def _redraw_all_vs_all(_=None):
    strategy_n = STRATEGY_OPTIONS[strategy_dropdown.value]
    all_vs_all, row_counts, col_counts = load_all_vs_all(Path(BASE_INDIR, STRATEGY_DIR[strategy_n]))
    _all_vs_all_out.clear_output(wait=True)
    with _all_vs_all_out:
        display(build_all_vs_all_figure(
            all_vs_all, row_counts, col_counts, _sort_by_dropdown.value,
            f"Strategy {strategy_n} agreement, collapsed categories (10 Ma samples pooled)",
        ))
       
_MARKERS_TO_PLOT = KAMATH_DA_MARKER_SYMBOLS

## 1 — All samples pooled: Ma (2025) agreement matrix/heatmap

In [ ]:
# Explicit Output()+observe() redraw loop -- widgets.interact() wasn't reliably
# clearing/redrawing plotly figures on dropdown change.
_sort_by_dropdown = dropdown(
    big=True,
    options=["counts (max to min)", "alphanumeric", "cluster"],
    description="Sort by:",
    layout=widgets.Layout(width="420px", height="56px"),
)
_all_vs_all_out = widgets.Output()

_sort_by_dropdown.observe(_redraw_all_vs_all, names="value")
strategy_dropdown.observe(_redraw_all_vs_all, names="value")
display(strategy_dropdown, _sort_by_dropdown, _all_vs_all_out)
_redraw_all_vs_all()

## 2 — Exploration of each sample (select a sample)
In the dropdowns that will appear below, select: i) a reference strategy (also used by Section 1 above), ii) a sample, and iii) either `All cell types` or `DAs only`

In [ ]:
display(strategy_dropdown)
display(sample_dropdown)
display(cell_type_view_dropdown)

In [ ]:
SAMPLE_NAME = sample_dropdown.value
STRATEGY_N = STRATEGY_OPTIONS[strategy_dropdown.value]
H5AD_PATH = Path(BASE_INDIR, STRATEGY_DIR[STRATEGY_N], SAMPLE_NAME + H5AD_SUFFIX)

print(f"Loading: {H5AD_PATH}")
adata = ad.read_h5ad(H5AD_PATH)
print(adata.shape)

cell_types = list(adata.obsm["q05_cell_abundance_w_sf"].columns)
leiden_keys = sorted(c for c in adata.obs.columns if "leiden_res_" in c)  # Strategy 1 uses "leiden_res_*"; Strategy 2/3 use "cell2location_leiden_res_*"
frac_cols = [f"frac_{ct}" for ct in cell_types]
cell_types_ = [strip_q05_abundance_prefix(ct) for ct in cell_types]
_ensure_uber_colors(cell_types_)

## 3 — Spatial cell type abundances
q05 posterior abundance per spot, one panel per cell type.

In [ ]:
for ct, ct_ in zip(cell_types, cell_types_):
    adata.obs[ct_] = adata.obsm["q05_cell_abundance_w_sf"][ct].values

cell_types_sel, cell_types_sel_ = selected_cell_types(cell_types, cell_types_, cell_type_view_dropdown.value, DA_UBER_CATEGORIES)

fig = spatial_scatter_grid(
    adata, cell_types_sel_,
    title_color_fn=lambda col: uber_palette[uber_type_of(col)],
)
if fig is not None:
    display(fig)
    plt.close(fig)

## 4 — Leiden clusters — UMAP and spatial

In [ ]:
sc.pl.spatial(
    adata,
    color=leiden_keys,
    alpha=0.6,
    ncols=4,
    size=1.3,
    img_key="hires",
)

leiden_key = leiden_keys[-1]  # highest resolution available; change the index to inspect others

mean_frac = (
    adata.obs[[leiden_key] + frac_cols]
    .groupby(leiden_key)[frac_cols]
    .mean()
    .rename(columns=strip_q05_abundance_prefix)
)

mean_frac.columns = pd.MultiIndex.from_tuples(
    [(uber_type_of(ct), ct) for ct in mean_frac.columns],
    names=["uber_type", "cell_type"],
)
col_colors = [uber_palette[u] for u in mean_frac.columns.get_level_values("uber_type")]

fig, ax = plt.subplots(figsize=(12, max(4, len(mean_frac) * 0.5)))
mean_frac.plot(kind="barh", stacked=True, ax=ax, color=col_colors, legend=False)
ax.set_xlabel("Mean fraction")
ax.set_ylabel(leiden_key)
ax.set_title(f"Cell type composition per Leiden cluster ({leiden_key})")

legend_handles = [mpatches.Patch(color=uber_palette[u], label=u) for u in uber_categories]
ax.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, title="Cell type")

plt.tight_layout()
plt.show()

## 5 — Cell type makers

In [ ]:
# Need to load the H5AD's with all genes (those from cell2location are HVG filtered)
RAW_SPATIAL_PATH = Path(BASE_INDIR, "spatial_h5ad", SAMPLE_NAME + ".spatial.h5ad")
print(f"Loading: {RAW_SPATIAL_PATH}")
raw_adata = ad.read_h5ad(RAW_SPATIAL_PATH)
raw_adata.var_names_make_unique()

markers_present = [g for g in _MARKERS_TO_PLOT if g in raw_adata.var_names]
markers_missing = [g for g in _MARKERS_TO_PLOT if g not in raw_adata.var_names]
if markers_missing:
    print(f"WARNING: {len(markers_missing)} marker(s) not in {SAMPLE_NAME}'s gene panel: {markers_missing}")
print(f"Plotting {len(markers_present)}/{len(_MARKERS_TO_PLOT)} markers")

sc.pp.normalize_total(raw_adata, target_sum=1e4)
sc.pp.log1p(raw_adata)

fig = spatial_scatter_grid(raw_adata, markers_present, cmap="viridis", vmax_floor=0.01)
if fig is not None:
    display(fig)
    plt.close(fig)

## 6 — Cell type co-occurrence (fraction correlation)

Pearson correlation of per-spot cell type fractions. High correlation = frequently co-localised.

In [ ]:
fractions = adata.obsm["cell_type_fractions"].rename(columns=strip_q05_abundance_prefix)
corr = fractions.corr()

fig, ax = plt.subplots(figsize=(max(8, len(cell_types_) * 0.6),
                                max(6, len(cell_types_) * 0.5)))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.7},
)
ax.set_title("Cell type fraction correlation across spots")

for tick_label in ax.get_xticklabels():
    tick_label.set_color(uber_palette[uber_type_of(tick_label.get_text())])
for tick_label in ax.get_yticklabels():
    tick_label.set_color(uber_palette[uber_type_of(tick_label.get_text())])

plt.tight_layout()
plt.show()

## 7 — Comparison vs. Ma's RCTD labels

Per-spot RCTD cell-type proportions from Ma et al. 2025 (see
`03_build_benchmark.py`).

In [ ]:
MA_BENCHMARK_CSV = str(Path(BASE_INDIR, "per_spot_benchmark_al_dalahmah_sn.csv.gz"))

ma_bench = pd.read_csv(MA_BENCHMARK_CSV)
ma_sample_id = SAMPLE_NAME.replace("-", "")  # benchmark table uses "T0081"; SAMPLE_NAME is "T-0081"
ma_sample = ma_bench[ma_bench["sample"] == ma_sample_id].set_index("barcode")

missing = set(adata.obs_names) - set(ma_sample.index)
if missing:
    print(f"WARNING: {len(missing)}/{adata.n_obs} spots have no Ma (2025) benchmark row")
ma_sample = ma_sample.reindex(adata.obs_names)

for col in _compare_mod.MA_CELL_TYPE_COLS:
    adata.obs[f"Ma_{col}"] = ma_sample[col].values

n_joined = ma_sample[_compare_mod.MA_CELL_TYPE_COLS].notna().all(axis=1).sum()
print(f"Joined {n_joined}/{adata.n_obs} spots to Ma (2025) RCTD proportions")

fig = spatial_scatter_grid(
    adata, [f"Ma_{col}" for col in _compare_mod.MA_CELL_TYPE_COLS],
    titles=list(_compare_mod.MA_CELL_TYPE_COLS),
)
if fig is not None:
    display(fig)
    plt.close(fig)